# Midden-laag spanning Stedin met maximale waterdiepte

In [1]:
from pathlib import Path
import geopandas as gpd
import folium 
from shapely.geometry import LineString, Polygon, box
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import shape

import numpy as np
import geopandas as gpd
from scipy.spatial import Voronoi
from shapely.geometry import Polygon

import rasterio
import geopandas as gpd
from shapely.geometry import Point
from rasterio.transform import Affine
import lonboard as lb

In [14]:
root_dir = Path(r'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse')
static_path = root_dir.joinpath("static_maxwd")
network_path = static_path.joinpath("network")
stedin_data = root_dir.joinpath("Stedin_data")
hazard_path = static_path.joinpath("hazard")
resultaten_path = static_path.joinpath("resultaten")
assert root_dir.exists()

In [3]:
msls_station_path = stedin_data.joinpath("MiddenLaagspanningsstations")
msls_stations = gpd.read_file(msls_station_path.joinpath("MiddenLaagspanningsstations.shp"), driver='ESRI Shapefile')
msls_stations_centroids = msls_stations['geometry'].representative_point()
bounding_box_msls = msls_stations_centroids.unary_union.convex_hull


In [4]:
boundary = gpd.read_file(network_path.joinpath("stedin_area.geojson"),driver='GeoJSON')

In [5]:
study_area = gpd.read_file(network_path.joinpath("study_area_correct_geometry.gpkg"),driver='GPKG')


In [6]:
study_area

,gml_id,nationalCo,localId,namespace,nationalLe,national_1,country,name,geometry
0,NL.WBHCODE.40.Admingrenswaterschap.1,40,Admingrenswaterschap.1,NL.WBHCODE.40,4thOrder,Waterschap,NL,Noord-Westelijke Delta,"MULTIPOLYGON (((102132.491 433603.397, 102146...."


In [7]:
def voronoi_polygons(gdf, bounding_box_gdf, boundary=None):
    from shapely.geometry import Polygon
    from scipy.spatial import Voronoi
    import geopandas as gpd

    # Combineer geometrieën in bounding_box_gdf tot één polygon
    bounding_geom = bounding_box_gdf.unary_union

    # Genereer Voronoi-diagram
    points = gdf.geometry.apply(lambda geom: (geom.x, geom.y)).tolist()
    vor = Voronoi(points)
    polygons = []

    for region in vor.regions:
        if not -1 in region and len(region) > 0:
            polygon = Polygon([vor.vertices[i] for i in region])
            clipped_polygon = polygon.intersection(bounding_geom)
            polygons.append(clipped_polygon)

    voronoi_gdf = gpd.GeoDataFrame(geometry=polygons, crs=gdf.crs)

    # Clip met boundary als die is meegegeven
    if boundary is not None:
        voronoi_gdf = voronoi_gdf.clip(boundary)

    return voronoi_gdf


# Assuming 'gdf' is your GeoDataFrame with point geometries

gdf_msls = msls_stations_centroids.set_crs("EPSG:28992")
voronoi_gdf_msls = voronoi_polygons(gdf_msls, study_area, boundary=boundary)
voronoi_gdf_msls = voronoi_gdf_msls.set_crs("EPSG:28992")




In [8]:
voronoi_gdf_msls.to_file(resultaten_path.joinpath("voronoi_gdf_msls.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 9887 WHERE lower(table_name) = lower('voronoi_gdf_msls')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 9887 WHERE lower(table_name) = lower('voronoi_gdf_msls')) failed: unable to open database file"


In [15]:
import rasterio
import matplotlib.pyplot as plt

hazard_map = hazard_path.joinpath("RMM_merge_max_waterdepth.tif")
# Open the TIF file using rasterio
with rasterio.open(hazard_map) as src:
    # Read the TIF file as a numpy array
    tif_array = src.read(1)  # Change the band index (1) if necessary


hazard_map_duration = hazard_path.joinpath("RMM_merge_duurkaart_uur.tif")
with rasterio.open(hazard_map_duration) as src:
    # Read the TIF file as a numpy array
    tif_array = src.read(1)  # Change the band index (1) if necessary


In [11]:
# Assuming gdf_msls is already a GeoDataFrame or a DataFrame with geometry data
gdf_msls_2 = gpd.GeoDataFrame()
gdf_msls_2['geometry'] = gdf_msls.geometry

C:\Users\meije_le\AppData\Local\Temp\ipykernel_26000\219393971.py:3: FutureWarning: You are adding a column named 'geometry' to a GeoDataFrame constructed without an active geometry column. Currently, this automatically sets the active geometry column to 'geometry' but in the future that will no longer happen. Instead, either provide geometry to the GeoDataFrame constructor (GeoDataFrame(... geometry=GeoSeries()) or use `set_geometry('geometry')` to explicitly set the active geometry column.
  gdf_msls_2['geometry'] = gdf_msls.geometry


In [ ]:
import rasterio
import geopandas as gpd

# Ensure CRS match between points and study area
points = gdf_msls_2.copy()
study_area = study_area.copy()

# Clip points to study area (ensure valid geometry)
study_area = study_area.buffer(0)
points_clipped = gpd.clip(points, study_area)

# Sample hazard map
with rasterio.open(hazard_map) as src_hazard:
    hazard_crs = src_hazard.crs
    points_clipped_hazard = points_clipped.to_crs(hazard_crs)
    coords_hazard = [(geom.x, geom.y) for geom in points_clipped_hazard.geometry]
    hazard_values = list(src_hazard.sample(coords_hazard))
    points_clipped['hazard_value'] = [val[0] if val else None for val in hazard_values]

# Sample duration map
with rasterio.open(hazard_map_duration) as src_duration:
    duration_crs = src_duration.crs
    points_clipped_duration = points_clipped.to_crs(duration_crs)
    coords_duration = [(geom.x, geom.y) for geom in points_clipped_duration.geometry]
    duration_values = list(src_duration.sample(coords_duration))
    points_clipped['duration_value'] = [val[0] if val else None for val in duration_values]

# Result: points_clipped now contains both hazard and duration values


In [23]:
import geopandas as gpd

def assign_hazard_to_voronoi(points_with_hazard, voronoi_gdf_msls):
    # Ensure both GeoDataFrames have the same CRS
    if points_with_hazard.crs != voronoi_gdf_msls.crs:
        points_with_hazard = points_with_hazard.to_crs(voronoi_gdf_msls.crs)
    
    # Spatial join: find Voronoi polygons that contain points
    joined_gdf = gpd.sjoin(voronoi_gdf_msls, points_with_hazard, how="inner", predicate="contains")
    
    # Group by Voronoi polygon and aggregate both hazard and duration values
    aggregated = joined_gdf.groupby(joined_gdf.index).agg({
        'hazard_value': 'mean',
        'duration_value': 'mean'
    }).reset_index()
    
    # Merge aggregated values back to the original Voronoi GeoDataFrame
    voronoi_gdf_msls = voronoi_gdf_msls.merge(aggregated, left_index=True, right_on='index', how='left')
    
    return voronoi_gdf_msls


In [ ]:
result = assign_hazard_to_voronoi(points_clipped, voronoi_gdf_msls)
result.to_file(resultaten_path.joinpath("msls_voronoi_hazard.gpkg"), driver='GPKG')

# Damage functions

In [35]:
def default_damage_ratio_function(hazard_values, coefficients=(0.0468, 0.0077)):
    """Calculate damage ratio from hazard values using linear function"""
    m, n = coefficients
    return m * hazard_values + n
 
points_clipped['damage_ratio'] = default_damage_ratio_function(points_clipped['hazard_value'])

c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [36]:
points_clipped['damage'] = points_clipped['damage_ratio'] * 142500 #construction costs

In [37]:
def default_repair_time_function(damage_ratios,coefficients=(702.72, 3.14, 1.9891)):
    """Calculate repair time from damage ratios using polynomial function"""
    a, b, c = coefficients
    return a * (damage_ratios ** 2) + b * damage_ratios + c


points_clipped['repair_time'] = default_repair_time_function(points_clipped['damage_ratio'])

c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\geopandas\geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [38]:
points_clipped 

,geometry,hazard_value,duration_value,damage,damage_ratio,repair_time
13000,POINT (101789.979 415234.583),NaN,NaN,NaN,NaN,NaN
13346,POINT (100898.892 415596.810),NaN,NaN,NaN,NaN,NaN
10121,POINT (100601.076 416034.702),NaN,NaN,NaN,NaN,NaN
16976,POINT (101602.073 417205.330),NaN,NaN,NaN,NaN,NaN
8596,POINT (100919.963 417205.826),NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
7103,POINT (96064.655 453681.252),NaN,NaN,NaN,NaN,NaN
16335,POINT (96627.923 453729.159),NaN,NaN,NaN,NaN,NaN
16245,POINT (96463.223 453811.012),NaN,NaN,NaN,NaN,NaN
16570,POINT (96366.200 453853.242),NaN,NaN,NaN,NaN,NaN


In [40]:
points_clipped.to_file(resultaten_path.joinpath("msls_stations_with_hazard.gpkg"), driver='GPKG')

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 48630.17072560786, min_y = 408543.478, max_x = 118567.7790863455, max_y = 459347.5544999987 WHERE lower(table_name) = lower('msls_stations_with_hazard') AND Lower(data_type) = 'features') failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 48630.17072560786, min_y = 408543.478, max_x = 118567.7790863455, max_y = 459347.5544999987 WHERE lower(table_name) = lower('msls_stations_with_hazard') AND Lower(data_type) = 'features') failed: disk I/O error"


## Impact per municipality

In [30]:
municipality = gpd.read_file(network_path.joinpath("gemeentegrenzen.shp"),driver='Esri Shapefile')
municipality_study_area = municipality.clip(study_area)

# BOXPLOT

With higher and lower bound construction costs

# 